# RTE sweep A: fixing distillation (DeBERTa-v3-xsmall)

The first distilled student (76.8) trailed the plain student (77.7): the teacher's soft labels lean to entailment and are confidently wrong on 17% of train. Two fixes, 5 seeds each, everything else identical to the plain student.

**Unattended:** Runtime → Change runtime type → **GPU (T4)**, then **Runtime → Run all**. Nothing to upload, no Google Drive.
Code is cloned from the public repo; data is rebuilt from public GLUE RTE (fingerprint-checked against the official file);
teacher logits ship in `artifacts/teacher/`. Expected time on a T4: **about 2–2.5 hours**.

Results are downloaded by the browser as soon as each run finishes (a small zip per run, plus `all_results.zip` at the end),
so **keep this tab open**, and allow automatic downloads for colab.research.google.com in Chrome beforehand
(chrome://settings/content/automaticDownloads), or Chrome will pause on a "download multiple files" prompt.
Checkpoints are not downloaded; every number is in the logs and zips.

## Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys
%cd /content
if os.path.isdir("/content/rte-challenge/.git"):
    !git -C /content/rte-challenge pull -q --ff-only
else:
    !git clone -q https://github.com/ihemanthm/rte-challenge.git
%cd /content/rte-challenge
!git log --oneline -1
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!pip install -q --no-deps -e .
# train.jsonl is rebuilt from public GLUE RTE and checked against the official file's fingerprint
!python scripts/prepare_data.py --out data

In [ ]:
import json, os, subprocess
from google.colab import files

def run(cmd: str, run_name: str) -> None:
    """Train, stream the log, then download outputs/<run_name> (no checkpoint) as a small zip.
    A failure is reported and the notebook moves on to the next run."""
    print(f">> {cmd}", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        print(f"!! {run_name} failed (exit {p.returncode}); continuing", flush=True)
    m = f"outputs/{run_name}/metrics.json"
    if os.path.exists(m):
        r = json.load(open(m))
        print(f"RESULT {run_name}: dev {r['dev_acc_mean']*100:.1f} ± {r['dev_acc_std']*100:.1f}, "
              f"params {r['params']['total']/1e6:.1f}M", flush=True)
    if os.path.isdir(f"outputs/{run_name}"):
        z = f"{run_name}.zip"
        subprocess.run(f"zip -q -r {z} outputs/{run_name} -x 'outputs/{run_name}/model/*'", shell=True)
        files.download(z)

## A1. Masked KD: gold-label CE only where the teacher is wrong (alpha 0.5, T 2)

In [ ]:
run("python -m rte.distill --config configs/distill_masked_deberta_v3_xsmall.yaml", "distill_masked_deberta_v3_xsmall")

## A2. Re-weighted KD: alpha 0.8, T 4

In [ ]:
run("python -m rte.distill --config configs/distill_a08_t4_deberta_v3_xsmall.yaml", "distill_a08_t4_deberta_v3_xsmall")

## Summary and download everything

In [ ]:
import glob
for f in sorted(glob.glob("outputs/*/metrics.json")):
    r = json.load(open(f)); p = r["params"]
    print(f"{r['run_name']:36s} dev {r['dev_acc_mean']*100:5.1f} ± {r['dev_acc_std']*100:4.1f}  "
          f"val {sum(s['best_val_acc'] for s in r['seeds'])/len(r['seeds'])*100:5.1f}  "
          f"params {p['total']/1e6:6.1f}M (emb {p['word_embedding']/1e6:5.1f}M)")
!zip -q -r all_results.zip outputs -x 'outputs/*/model/*'
files.download("all_results.zip")